# Exercise 2.2.3: Cleaning, Missing Values & Duplicates
*Exercício 2.2.3: Limpeza, Valores em Falta e Duplicados*

This notebook continues from the **typed checkpoint** produced by 2.2.2 (`data/10_cleaned/datania_households_clean.csv`). Types are already fixed: income is numeric, dates are parsed, category typos are corrected. Now you handle the messiness that *types alone don't fix*: missing values, coded/sentinel values, duplicates, and impossible values.

*Este notebook continua a partir do **ponto de controlo com tipos corrigidos** produzido no 2.2.2 (`data/10_cleaned/datania_households_clean.csv`). Os tipos já estão corrigidos: o rendimento é numérico, as datas estão convertidas e os erros de escrita nas categorias estão corrigidos. Agora vai tratar dos problemas que *os tipos por si só não resolvem*: valores em falta, códigos e valores sentinela, duplicados e valores impossíveis.*

You will practice:
- Working on a **copy** and documenting every change
- Detecting missing values with `isna().sum()`, percentages, and row inspection
- Recoding **coded / sentinel missing values** (`99`, `999`, `9999`, `999999`) and recovering sign-entry errors with `.abs()`
- Choosing a missing-value strategy: drop, drop a subset, or fill
- Detecting and removing **exact** and **subset** duplicates
- Applying **validation rules** to catch impossible values
- Saving the fully cleaned dataset back to `10_cleaned/`

*Vai praticar:*
- *Trabalhar sobre uma **cópia** e documentar cada alteração*
- *Detetar valores em falta com `isna().sum()`, percentagens e inspeção de linhas*
- *Recodificar **códigos e valores sentinela de ausência** (`99`, `999`, `9999`, `999999`) e recuperar erros de sinal com `.abs()`*
- *Escolher uma estratégia para os valores em falta: eliminar, eliminar apenas num subconjunto de colunas, ou preencher*
- *Detetar e remover duplicados **exatos** e **por subconjunto de colunas***
- *Aplicar **regras de validação** para apanhar valores impossíveis*
- *Guardar o conjunto de dados totalmente limpo de novo em `10_cleaned/`*

> **Pipeline:** reads the typed checkpoint from `10_cleaned/` and overwrites it with the cleaned dataset. Exercise 2.2.4 reads from there.

> ***Fluxo de trabalho:** lê o ponto de controlo de `10_cleaned/` e substitui-o pelo conjunto de dados limpo. O Exercício 2.2.4 lê a partir daí.*

### Path Setup (run first)
*Configuração do caminho (execute primeiro)*

In [ ]:
import os
import numpy as np
import pandas as pd

DATA_CLEAN_DIR = '../../data/10_cleaned'
clean_path = os.path.join(DATA_CLEAN_DIR, 'datania_households_clean.csv')

# Typed checkpoint from 2.2.2: income is numeric, dates parsed, category typos fixed
# Ponto de controlo do 2.2.2: rendimento numérico, datas convertidas, erros de categoria corrigidos
df = pd.read_csv(clean_path, dtype={'hh_id': str, 'region_code': str})

print('Loaded typed checkpoint:', df.shape)
df.head()

---

## Task 1: Never destroy the input: make a clean copy
*Tarefa 1: Nunca destrua os dados de entrada: faça uma cópia*

Cleaning operations should never touch the DataFrame you loaded. Create a separate working copy and apply every change to it. This keeps the checkpoint intact for comparison and lets you restart if your assumptions change.

*As operações de limpeza nunca devem tocar no DataFrame que carregou. Crie uma cópia de trabalho separada e aplique nela todas as alterações. Assim o ponto de controlo mantém-se intacto para comparação e pode recomeçar se os seus pressupostos mudarem.*

In [ ]:
# Create the working copy | Criar a cópia de trabalho
df_clean = df.  # your code here | o seu código aqui

print('Working copy:', df_clean.shape)

**The golden rule of notebook documentation**

- **Above each code cell** (Markdown): explain *what* you are about to do and *why*.
- **Below each code cell** (Markdown): interpret the *result* and note the decision you made.

***A regra de ouro da documentação em notebooks***

- ***Acima de cada célula de código** (Markdown): explique *o que* vai fazer e *porquê*.*
- ***Abaixo de cada célula de código** (Markdown): interprete o *resultado* e registe a decisão que tomou.*

---

## Task 2: Detect missing values
*Tarefa 2: Detetar valores em falta*

`isna().sum()` counts blanks/`NaN` per column. The percentage version shows how serious each gap is, and inspecting the affected rows shows *which* records are incomplete. Because 2.2.2 already converted `income_dkw` and `survey_date`, their text placeholders (`unknown`, `not recorded`) are already real `NaN`/`NaT` here.

*`isna().sum()` conta os vazios/`NaN` por coluna. A versão em percentagem mostra a gravidade de cada lacuna e a inspeção das linhas afetadas mostra *quais* os registos incompletos. Como o 2.2.2 já converteu `income_dkw` e `survey_date`, os seus marcadores de texto (`unknown`, `not recorded`) já são aqui verdadeiros `NaN`/`NaT`.*

In [ ]:
# Count missing values per column | Contar os valores em falta por coluna
df_clean.  # your code here | o seu código aqui

In [ ]:
# Percentage missing per column (round to 2 decimals)
# Percentagem de valores em falta por coluna (arredondar a 2 casas decimais)
(df_clean.isna().sum() / # your code here | o seu código aqui ).round(2)

In [ ]:
# Show every row that has at least one missing value
# Mostrar todas as linhas com pelo menos um valor em falta
df_clean[df_clean.isna(). # your code here | o seu código aqui ]

**Questions:**

- Which columns have missing values? How many rows are affected?
- Which gaps look serious enough to worry about, and which are negligible?

***Perguntas:***

- *Que colunas têm valores em falta? Quantas linhas são afetadas?*
- *Que lacunas parecem suficientemente graves para preocupar e quais são negligenciáveis?*

---

## Task 3: Coded missing values
*Tarefa 3: Códigos de valores em falta*

Survey data mixes a few problems that all look like numbers:

*Os dados de inquérito misturam vários problemas que se parecem todos com números:*

| Value / Valor | Meaning / Significado | Fix / Correção |
|---|---|---|
| `-5000` | negative income, a **sign-entry error** / rendimento negativo, **erro de sinal** | recover with `.abs()` / recuperar com `.abs()` |
| `999999` | all-nines "not stated" **sentinel** / **sentinela** de noves "não declarado" | recode to `NaN` / recodificar para `NaN` |
| `999`, `9999`, `99` | column sentinels (age, density, education) / sentinelas de coluna (idade, densidade, educação) | recode to `NaN` / recodificar para `NaN` |

Not every odd value is missing: a negative income is a fixable typo whose magnitude is real, while a sentinel carries no real value. Handle each appropriately and check the impact.

*Nem todo o valor estranho é um valor em falta: um rendimento negativo é um erro corrigível cuja magnitude é real, enquanto uma sentinela não transporta qualquer valor real. Trate cada caso de forma adequada e verifique o impacto.*

In [ ]:
# A negative income is a sign-entry error: recover the magnitude (it is NOT missing)
# Um rendimento negativo é um erro de sinal: recupere a magnitude (NÃO é um valor em falta)
df_clean['income_dkw'] = df_clean['income_dkw'].  # your code here: .abs() | o seu código aqui: .abs()

# 999999 is an all-nines "not stated" sentinel: recode it to NaN
# 999999 é uma sentinela de noves para "não declarado": recodifique para NaN
df_clean['income_dkw'] = df_clean['income_dkw'].replace( # your code here: 999999, np.nan | o seu código aqui )

df_clean['income_dkw'].describe()

In [ ]:
# A reusable helper keeps the recoding consistent across columns
# Uma função auxiliar reutilizável mantém a recodificação coerente entre colunas
def recode_coded_missing(series, codes):
    """Replace coded missing values with NaN.
    Substituir os códigos de valores em falta por NaN."""
    return series.replace(codes, np.nan)

print('Mean age BEFORE recode:', round(df_clean['age'].mean(), 1))

df_clean['age'] = recode_coded_missing(df_clean['age'], [999])
df_clean['pop_density'] = recode_coded_missing(df_clean['pop_density'], # your code here: [9999] | o seu código aqui: [9999] )
df_clean['education_code'] = recode_coded_missing(df_clean['education_code'], # your code here: [99] | o seu código aqui: [99] )

print('Mean age AFTER recode: ', round(df_clean['age'].mean(), 1))

**Questions:**

- Why recover the negative income with `.abs()` but recode `999999` to `NaN`? What makes one a fixable error and the other truly missing?
- By how much did the mean age change after recoding `999`? What does that say about leaving sentinels in place?
- Why is `0` a *tricky* code (think `hh_size` or `income`)?

***Perguntas:***

- *Porquê recuperar o rendimento negativo com `.abs()` mas recodificar `999999` para `NaN`? O que faz de um um erro corrigível e do outro um valor verdadeiramente em falta?*
- *Quanto variou a idade média depois de recodificar `999`? O que é que isso diz sobre deixar as sentinelas nos dados?*
- *Porque é que `0` é um código *delicado* (pense em `hh_size` ou `income`)?*

---

## Task 4: Missing-value strategy: drop critical-only
*Tarefa 4: Estratégia para valores em falta: eliminar apenas os críticos*

Three strategies exist: drop any row with a gap, drop only rows missing a **critical** column, or **fill** the gaps. Dropping everything is usually too aggressive. Here the only non-negotiable column is the identifier `hh_id`: drop rows missing it, leave the rest.

*Existem três estratégias: eliminar qualquer linha com uma lacuna, eliminar apenas as linhas sem uma coluna **crítica**, ou **preencher** as lacunas. Eliminar tudo costuma ser demasiado agressivo. Aqui a única coluna inegociável é o identificador `hh_id`: elimine as linhas sem esse valor e deixe as restantes.*

In [ ]:
# Drop only rows without an identifier | Eliminar apenas as linhas sem identificador
print('Before:', df_clean.shape)
df_clean = df_clean.dropna(subset= # your code here | o seu código aqui )
print('After: ', df_clean.shape)

In [ ]:
# A fill is sometimes appropriate. Preview only (do NOT overwrite df_clean):
# Por vezes preencher é adequado. Apenas pré-visualização (NÃO substitua df_clean):
example = df_clean['income_dkw']. # your code here: fillna with the median | o seu código aqui: fillna com a mediana
print('NaN before fill:', df_clean['income_dkw'].isna().sum(),
      '| NaN after fill:', example.isna().sum())

**Questions:**

- Did `dropna(subset=['hh_id'])` remove any rows here? Why is it still worth running?
- Filling income with the median changes the distribution. When is that acceptable, and when does it introduce bias?

***Perguntas:***

- *`dropna(subset=['hh_id'])` removeu alguma linha neste caso? Porque vale a pena executá-lo mesmo assim?*
- *Preencher o rendimento com a mediana altera a distribuição. Quando é isso aceitável e quando é que introduz enviesamento?*

---

## Task 5: Detect and remove duplicates
*Tarefa 5: Detetar e remover duplicados*

Real surveys record the same household twice. First handle **exact** duplicates (every column identical), then **subset** duplicates (same `hh_id`, different other fields).

*Os inquéritos reais registam o mesmo agregado familiar duas vezes. Trate primeiro os duplicados **exatos** (todas as colunas iguais) e depois os duplicados **por subconjunto** (mesmo `hh_id`, restantes campos diferentes).*

In [ ]:
# How many fully-identical rows are there? Show them.
# Quantas linhas totalmente idênticas existem? Mostre-as.
print('Exact duplicate rows:', df_clean.duplicated().sum())
df_clean[df_clean.duplicated( # your code here: keep=False | o seu código aqui: keep=False )].sort_values('hh_id')

In [ ]:
# Remove the exact duplicates | Remover os duplicados exatos
print('Before:', df_clean.shape)
df_clean = df_clean. # your code here: drop_duplicates() | o seu código aqui: drop_duplicates()
print('After: ', df_clean.shape)

In [ ]:
# Subset duplicates: same hh_id, but the rows differ. Show all occurrences.
# Duplicados por subconjunto: mesmo hh_id, mas linhas diferentes. Mostre todas as ocorrências.
dups = df_clean[df_clean.duplicated(subset= # your code here | o seu código aqui , keep=False)]
dups.sort_values('hh_id')[['hh_id', 'district', 'income_dkw', 'survey_date']]

When two rows share an `hh_id` but differ, choose which to keep. A robust rule is **keep the most complete** record (the one with the fewest missing values).

*Quando duas linhas partilham o mesmo `hh_id` mas diferem, escolha qual manter. Uma regra robusta é **manter o registo mais completo** (o que tem menos valores em falta).*

In [ ]:
# Sort by completeness, then keep the first row of each hh_id
# Ordenar por grau de preenchimento e manter a primeira linha de cada hh_id
df_clean['missing_count'] = df_clean.isna().sum(axis=1)
df_clean = (
    df_clean
    .sort_values(['hh_id', 'missing_count'])
    .drop_duplicates(subset=['hh_id'], keep= # your code here: 'first' | o seu código aqui: 'first' )
)
df_clean = df_clean.drop(columns='missing_count')
print('After resolving subset duplicates:', df_clean.shape)

**Questions:**

- Which `hh_id` was an exact duplicate? Which was a subset duplicate?
- For the subset duplicate, which record was kept and why? What other rule could you use (hint: `survey_date`)?
- Why does `keep=False` matter when you are *inspecting* duplicates?

***Perguntas:***

- *Que `hh_id` era um duplicado exato? E qual era um duplicado por subconjunto?*
- *No duplicado por subconjunto, que registo foi mantido e porquê? Que outra regra poderia usar (sugestão: `survey_date`)?*
- *Porque é que `keep=False` é importante quando se está a *inspecionar* duplicados?*

---

## Task 6: Validation rules: catch impossible values
*Tarefa 6: Regras de validação: apanhar valores impossíveis*

Some values are not missing: they are *impossible*. A household cannot have `0`, `-1`, or `99` members. Filter out the rows that fail a plausibility rule, checking the shape before and after.

*Alguns valores não estão em falta: são *impossíveis*. Um agregado familiar não pode ter `0`, `-1` ou `99` membros. Filtre as linhas que falham uma regra de plausibilidade, verificando as dimensões antes e depois.*

In [ ]:
# Keep only plausible household sizes | Manter apenas dimensões plausíveis do agregado
print('hh_size values:', sorted(df_clean['hh_size'].unique()))
print('Before:', df_clean.shape)
df_clean = df_clean[df_clean['hh_size'].between( # your code here: 1, 20 | o seu código aqui: 1, 20 )]
print('After: ', df_clean.shape)

**Questions:**

- How many rows did the `hh_size` rule remove? Which households were they?
- `age` was already handled by sentinel recoding in Task 3. What is the difference between *recoding to NaN* and *dropping the row*?

***Perguntas:***

- *Quantas linhas foram removidas pela regra de `hh_size`? Que agregados eram?*
- *A coluna `age` já foi tratada pela recodificação de sentinelas na Tarefa 3. Qual é a diferença entre *recodificar para NaN* e *eliminar a linha*?*

---

## Task 7: Save the cleaned dataset
*Tarefa 7: Guardar o conjunto de dados limpo*

Overwrite the checkpoint in `10_cleaned/` with the fully cleaned result, then reload it to confirm it round-trips. (Raw data in `0_raw/` is never touched.)

*Substitua o ponto de controlo em `10_cleaned/` pelo resultado totalmente limpo e depois volte a carregá-lo para confirmar que está correto. (Os dados brutos em `0_raw/` nunca são tocados.)*

In [ ]:
# Final check before saving | Verificação final antes de guardar
print('Final cleaned shape:', df_clean.shape)
df_clean.isna().sum()

In [ ]:
df_clean = df_clean.reset_index(drop=True)
out_path = os.path.join(DATA_CLEAN_DIR, 'datania_households_clean.csv')

df_clean.to_csv( # your code here: index=False | o seu código aqui: index=False )
print('Saved:', out_path)

In [ ]:
# Reload to confirm it round-trips | Voltar a carregar para confirmar que está correto
check = pd.read_csv(out_path, dtype={'hh_id': str, 'region_code': str})
print('Reloaded shape:', check.shape)
print()
print(check.dtypes)
check.head()

**Questions:**

- How many rows survived the full cleaning pipeline, starting from the 28-row checkpoint?
- After reloading, what dtype does `survey_date` have? What would you do before using it for date analysis?
- Could you justify every removed row to a reviewer?

***Perguntas:***

- *Quantas linhas sobreviveram a todo o processo de limpeza, partindo do ponto de controlo com 28 linhas?*
- *Depois de voltar a carregar, que dtype tem `survey_date`? O que faria antes de o usar numa análise temporal?*
- *Conseguiria justificar cada linha removida perante um revisor?*